# Argus — RandomForest Training

A tabular baseline trained directly on single-frame features — no windowing, no sequence
modeling. Split out from what used to be `02_model_training.ipynb`'s RandomForest section, and
repointed at the new flat per-frame dataset instead of window-mean-aggregated features (see
`02_dataset_creation_flat.ipynb`'s "Why one CSV, one row per frame" note for why: this model
family can't see a sequence anyway, so training it on windows only added generation cost without
adding usable signal).

**Reads:** `dataset_processed/frame_features.csv`, written by
[`02_dataset_creation_flat.ipynb`](./02_dataset_creation_flat.ipynb) — run that first (it also
produces the Spearman-correlation feature ranking this notebook selects its input features from).

**Writes:** `models/rf_classifier_<VERSION>.joblib`, `models/rf_feature_scaler_<VERSION>.joblib`.

Note the `rf_` prefix on the scaler filename — it's deliberately **not** named
`feature_scaler_*.joblib` like `03_model_training_lstm.ipynb`'s scaler. Both notebooks' scalers
are fit on different feature sets (7 selected columns here vs. all 58 per-timestep features
there); `03_deployment_export.ipynb` looks up the latest `feature_scaler_*.joblib` for the LSTM
specifically, and giving this one a distinct prefix keeps it from ever being picked up by
mistake — this is exactly the kind of same-name-different-shape collision that used to exist
between the old RandomForest and LSTM scalers before RF moved out of that notebook.


## Setup

Mounts Drive and defines the paths/constants this notebook needs. Independent of any other notebook's in-memory state — only depends on `frame_features.csv`.


In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

project_folder = "/content/drive/MyDrive/Argus"
print(f"Google Drive successfully mounted! Base project directory: {project_folder}")


Mounted at /content/drive
Google Drive successfully mounted! Base project directory: /content/drive/MyDrive/Argus


In [2]:
import os
import pandas as pd

# --- Argus project paths (must match 02_dataset_creation_flat.ipynb) ---
models_folder = f"{project_folder}/models"
dataset_folder = f"{project_folder}/dataset"
processed_folder = f"{dataset_folder}/dataset_processed"
frame_features_csv_path = os.path.join(processed_folder, "frame_features.csv")

if not os.path.exists(frame_features_csv_path):
    raise FileNotFoundError(
        f"'{frame_features_csv_path}' not found. Run 02_dataset_creation_flat.ipynb first — "
        "this notebook only reads the dataset it produces, it doesn't build it."
    )

df_frame_features = pd.read_csv(frame_features_csv_path)
df_frame_features['EAR_mean'] = (df_frame_features['EAR_left'] + df_frame_features['EAR_right']) / 2

print(f"Loaded flat dataset: {len(df_frame_features)} frame rows, "
      f"{df_frame_features['subject'].nunique()} subjects.")


Loaded flat dataset: 173730 frame rows, 24 subjects.


In [3]:
import joblib
import datetime

# --- Model Management Configuration ---
FORCE_RETRAIN = True

VERSION_STR = datetime.datetime.now().strftime("%Y%m%d_%H%M")
rf_model_path = os.path.join(models_folder, f"rf_classifier_{VERSION_STR}.joblib")
rf_scaler_path = os.path.join(models_folder, f"rf_feature_scaler_{VERSION_STR}.joblib")

def get_latest_model(folder, prefix, extension):
    """Return the path of the most recently versioned model matching prefix/extension, or None."""
    if not os.path.exists(folder):
        return None
    files = [f for f in os.listdir(folder) if f.startswith(prefix) and f.endswith(extension)]
    if not files:
        return None
    return os.path.join(folder, sorted(files)[-1])

print(f"✅ Model management initialized. Current version: {VERSION_STR}. Force retrain: {FORCE_RETRAIN}")


✅ Model management initialized. Current version: 20260819_0023. Force retrain: True


## Feature Selection

Per `02_dataset_creation_flat.ipynb`'s Spearman correlation ranking, we select the features with
the strongest monotonic relationship to drowsiness level: eye-closure and squint blendshapes,
`EAR_mean` (their geometric counterpart), `jawOpen`, and `pitch` (head tilting forward is a
late-stage drowsiness cue). `MAR`/`mouthFunnel` are intentionally excluded — that analysis found
them weakly correlated per-frame, consistent with yawning being a dynamic, multi-frame event a
static per-frame ratio doesn't capture well.


In [4]:
selected_features = [
    'eyeBlinkLeft',
    'eyeBlinkRight',
    'EAR_mean',
    'eyeSquintLeft',
    'eyeSquintRight',
    'jawOpen',
    'pitch',
]

X = df_frame_features[selected_features]
y = df_frame_features['level']
groups = df_frame_features['subject']  # for the group-aware split below

print(f"Selected features for Random Forest: {selected_features}")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")


Selected features for Random Forest: ['eyeBlinkLeft', 'eyeBlinkRight', 'EAR_mean', 'eyeSquintLeft', 'eyeSquintRight', 'jawOpen', 'pitch']
Shape of X: (173730, 7), Shape of y: (173730,)


## Splitting and Scaling Data

Group-aware 80/20 split so no subject appears in both train and test — frames from the same clip are highly correlated with each other, so a plain random/stratified split would leak near-duplicate samples across the split and inflate reported accuracy, exactly like the bug already fixed in the LSTM's split.


In [5]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_scaled, y, groups=groups))
X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train set size: {len(X_train)} frames ({groups.iloc[train_idx].nunique()} subjects)")
print(f"Test set size: {len(X_test)} frames ({groups.iloc[test_idx].nunique()} subjects)")
print("Subjects are disjoint between training and testing sets (group-aware split).")

joblib.dump(scaler, rf_scaler_path)
print(f"💾 Scaler saved: {rf_scaler_path}")


Train set size: 136106 frames (19 subjects)
Test set size: 37624 frames (5 subjects)
Subjects are disjoint between training and testing sets (group-aware split).
💾 Scaler saved: /content/drive/MyDrive/Argus/models/rf_feature_scaler_20260819_0023.joblib


## Training the Random Forest Model

A `RandomForestClassifier`, with `class_weight='balanced'` for the same reason the LSTM now uses `class_weight` (see `03_model_training_lstm.ipynb`): the `Drowsy` class is plausibly the rarest in the dataset — it's the one level that had to be partly acted rather than self-recorded — and it's also the single most safety-critical class to not neglect.


In [6]:
import os
from sklearn.ensemble import RandomForestClassifier

latest_rf = get_latest_model(models_folder, "rf_classifier", ".joblib")
latest_rf_scaler = get_latest_model(models_folder, "rf_feature_scaler", ".joblib")

if not FORCE_RETRAIN and latest_rf and latest_rf_scaler:
    print(f"📦 Loading existing Random Forest model: {latest_rf}")
    rf_classifier = joblib.load(latest_rf)
    scaler = joblib.load(latest_rf_scaler)

    # Re-scale using the loaded scaler, reusing the SAME group-aware split indices computed
    # above (not a fresh split) so reloading a model doesn't reintroduce subject leakage.
    X_re_scaled = scaler.transform(X)
    X_train, X_test = X_re_scaled[train_idx], X_re_scaled[test_idx]
else:
    print("🚀 Training new Random Forest Model...")
    rf_classifier = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    )
    rf_classifier.fit(X_train, y_train)
    joblib.dump(rf_classifier, rf_model_path)
    print(f"💾 Model saved: {rf_model_path}")


🚀 Training new Random Forest Model...
💾 Model saved: /content/drive/MyDrive/Argus/models/rf_classifier_20260819_0023.joblib


## Evaluating the Random Forest Model

A classification report on the held-out test set, plus feature importances to sanity-check that the model is actually leaning on the features the correlation analysis flagged as predictive.


In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

y_pred = rf_classifier.predict(X_test)

CLASS_NAMES = ['Not Drowsy', 'Drowsy']
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

overall_accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {overall_accuracy:.4f}")

conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (Random Forest)')
plt.show()

importances = rf_classifier.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': selected_features, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

print("\nFeature Importances:")
print(feature_importance_df)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
plt.title('Random Forest Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


### Reading this result

Because each row is a single frame rather than a window, expect this model's accuracy to sit
**below** whatever the original window-mean-aggregated RandomForest reported — averaging over a
window smooths out frame-level jitter the same way it does in the correlation analysis, and this
model doesn't get that smoothing. That's an expected, documentable tradeoff for the titulación
report (much cheaper dataset generation, at the cost of per-frame noise), not a regression to
chase down.


## Diagnosing the Low Accuracy

This is a **pre-migration 3-class run; no binary run yet.** That earlier run came back only
a little above chance, with `Drowsy` recall collapsing to near-zero — too low to be explained
by the expected per-frame-vs-window noise penalty (see "Reading this result" above) alone. The
two suspected causes below are structural and still apply under binary labels, so the
diagnostic cells that follow are kept:

1. **Subject-level baseline shift.** Raw EAR/pitch/blendshape values carry large *between-person*
   differences (eye shape, camera angle, resting expression) that can swamp the drowsiness
   signal. With few subjects total (see the split printout above), the model can partly learn
   "whose face is this" instead of "how drowsy is this face," which collapses on unseen subjects.
2. **Unregularized overfitting.** `max_depth=None` lets trees grow to pure leaves over just 7
   features — a strong recipe for memorizing training-subject idiosyncrasies instead of a
   generalizable pattern.

The cells below test both directly, rather than guessing: a train-vs-test accuracy check
(distinguishes overfitting from a signal-absence problem), a properly regularized/tuned RF via
subject-grouped cross-validation, and an optional per-subject-normalized feature variant that
directly tests hypothesis 1.


### Step 1 — Train vs. Test Accuracy (Overfitting Check)

If train accuracy is high while test accuracy is low, that's overfitting (fix: regularize — Step 2). If train accuracy is *also* low, the raw features aren't carrying enough subject-invariant signal at this depth of regularization, and Step 3 (per-subject normalization) is the more relevant fix.


In [ ]:
from sklearn.metrics import accuracy_score

train_accuracy = accuracy_score(y_train, rf_classifier.predict(X_train))
test_accuracy = accuracy_score(y_test, rf_classifier.predict(X_test))

print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Test accuracy:  {test_accuracy:.4f}")
print(f"Gap:            {train_accuracy - test_accuracy:.4f}")

if train_accuracy - test_accuracy > 0.15:
    print("\n⚠️  Large train/test gap -> overfitting is a real contributor. Step 2 should help.")
else:
    print("\n⚠️  Small train/test gap despite low test accuracy -> the model isn't finding much "
          "signal even on training data at this regularization level; look hard at Step 3.")


### Step 2 — Regularized RandomForest via Subject-Grouped Hyperparameter Search

`RandomizedSearchCV` over regularization parameters (`max_depth`, `min_samples_leaf`,
`min_samples_split`, `max_features`), scored on `f1_macro` (penalizes per-class recall
collapse — the `Drowsy` failure mode seen in the pre-migration 3-class run) rather than raw
accuracy. `f1_macro` is kept for binary too, so this number stays comparable with every other
model in the project. Cross-validation uses `GroupKFold` grouped by subject — the same reason
every split in this project is group-aware: without it, the search itself would leak subjects
across its internal folds and report optimistic scores that don't hold up on the real held-out
test set.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.ensemble import RandomForestClassifier

param_distributions = {
    'n_estimators': [200, 400, 600],
    'max_depth': [4, 6, 8, 10, 15, None],
    'min_samples_leaf': [1, 5, 20, 50, 100],
    'min_samples_split': [2, 10, 50, 100],
    'max_features': ['sqrt', 'log2', None],
}

group_kfold = GroupKFold(n_splits=5)
search = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=25,               # raise this if you have time/compute budget for a wider search
    scoring='f1_macro',
    cv=group_kfold,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train, groups=groups.iloc[train_idx])

print(f"\nBest params: {search.best_params_}")
print(f"Best CV f1_macro: {search.best_score_:.4f}")

rf_tuned = search.best_estimator_


In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_pred_tuned = rf_tuned.predict(X_test)

print("--- Tuned RandomForest (plain-scaled features) ---")
print(classification_report(y_test, y_pred_tuned, target_names=CLASS_NAMES))
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")

conf_matrix_tuned = confusion_matrix(y_test, y_pred_tuned)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_tuned, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (Tuned RandomForest)')
plt.show()


### Step 3 — Per-Subject-Normalized Features (Optional, Tests Hypothesis 1 Directly)

Each feature is z-scored **within each subject** (using that subject's own mean/std across all
their frames — this only uses each subject's own data, not labels, so it isn't a leakage risk
the way label-informed normalization would be). This removes between-person baseline differences
and leaves only *relative deviation from that person's own baseline* — which is what should
actually correlate with drowsiness, if hypothesis 1 is right.

**Deployment caveat, worth stating plainly in the titulación report if this is adopted:**
per-subject normalization requires knowing a subject's own baseline distribution, which a brand
-new driver doesn't have on their first trip. Adopting this for real deployment (not just as a
diagnostic here) would need a calibration step (e.g., the first N seconds/minutes of a trip
establish a rolling personal baseline) — a real design addition, not something this dataset
notebook currently provides.


In [ ]:
import numpy as np

df_norm = df_frame_features.copy()
df_norm[selected_features] = df_norm.groupby('subject')[selected_features].transform(
    lambda s: (s - s.mean()) / (s.std() + 1e-8)
)

X_subj = df_norm[selected_features]
# Reuse the exact same group-aware split indices as the main run above, so this is a fair
# apples-to-apples comparison against rf_tuned rather than a differently-split evaluation.
X_subj_train, X_subj_test = X_subj.iloc[train_idx].to_numpy(), X_subj.iloc[test_idx].to_numpy()

rf_subject_norm = RandomForestClassifier(**search.best_params_, class_weight='balanced', random_state=42, n_jobs=-1)
rf_subject_norm.fit(X_subj_train, y_train)

y_pred_subj = rf_subject_norm.predict(X_subj_test)

print("--- Tuned RandomForest (per-subject-normalized features) ---")
print(classification_report(y_test, y_pred_subj, target_names=CLASS_NAMES))
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred_subj):.4f}")

conf_matrix_subj = confusion_matrix(y_test, y_pred_subj)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_subj, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (Tuned RandomForest, Per-Subject-Normalized)')
plt.show()


### Reading the three results together

Compare the original curated-7-feature baseline, `rf_tuned` (plain-scaled, regularized), and
`rf_subject_norm` (per-subject-normalized, regularized) — on whatever the binary run reports,
since there is no binary number yet:

* **If `rf_tuned` improves a lot over the baseline** → overfitting was the dominant problem, and
  the regularized hyperparameters from Step 2 are your answer; per-subject normalization is
  optional polish, not required.
* **If `rf_subject_norm` clearly beats `rf_tuned`** → subject-level baseline shift was the
  dominant problem. That's a genuine, reportable finding for the titulación report — it says raw
  per-frame geometric features don't transfer across people well, which is exactly the kind of
  thing a windowed/temporal model (the LSTM) or a calibration step can address but a single
  frame's absolute values structurally can't.
* **If neither moves the needle much** → the ceiling here is more likely the fundamental
  per-frame information limit already flagged in "Reading this result" above (a single frame
  genuinely can't see duration/velocity-of-eye-closure), not something further RF tuning can fix
  — that's the strongest argument for the LSTM being the right model for deployment, not this one.


## Feature & Data Augmentation: Enriched Features, HistGradientBoosting, SMOTE

Three further experiments, layered on top of everything above rather than replacing it, so the
original curated-7-feature baseline and the tuned/per-subject-normalized variants stay available
for comparison:

1. **All (58 raw + rolling) features from `frame_features_enriched.csv`**, instead of the 7
   manually curated ones — trees can find combined/nonlinear signal across many individually-weak
   features that a univariate Spearman ranking can't see, and the rolling features from
   `02_dataset_creation_flat.ipynb`'s enrichment section carry real temporal information the
   original per-frame features structurally can't.
2. **`HistGradientBoostingClassifier`** on the same enriched feature set — boosted trees are
   often meaningfully stronger than RandomForest on tabular data.
3. **SMOTE** oversampling applied to the training set only, mainly to firm up `Drowsy` recall.

**Calibrated expectation:** none of these raise the *information* ceiling the way the rolling
features already might have — they help extract more of whatever signal is actually present.
Don't expect this section alone to reach 60-70%; if the enriched features carry meaningfully more
signal than the raw ones (check the "Sanity Check" cell in `02_dataset_creation_flat.ipynb`),
that's where a real jump would come from, not from swapping algorithms.


### Load the Enriched Dataset

A fresh group-aware split is computed here rather than reusing `train_idx`/`test_idx` from above -- `frame_features_enriched.csv` is sorted differently (by clip/frame order, for the rolling computation), so row positions don't line up with the original split. Same `random_state=42` and grouping by subject means the *same set of subjects* ends up in train/test either way, just at different row positions.


In [ ]:
import pandas as pd

enriched_csv_path = os.path.join(processed_folder, "frame_features_enriched.csv")
if not os.path.exists(enriched_csv_path):
    raise FileNotFoundError(
        f"'{enriched_csv_path}' not found. Run the \"Temporal Feature Enrichment\" section at "
        "the end of 02_dataset_creation_flat.ipynb first."
    )

df_enriched = pd.read_csv(enriched_csv_path)
enriched_id_cols = ['subject', 'level', 'parent_video', 'frame_idx']
enriched_feature_cols = [c for c in df_enriched.columns if c not in enriched_id_cols]

X_enriched = df_enriched[enriched_feature_cols].to_numpy(dtype=np.float32)
y_enriched = df_enriched['level'].to_numpy()
groups_enriched = df_enriched['subject'].to_numpy()

print(f"Enriched feature count: {len(enriched_feature_cols)} (vs. {len(selected_features)} curated)")
print(f"X_enriched shape: {X_enriched.shape}")


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

gss_enriched = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx_enr, test_idx_enr = next(gss_enriched.split(X_enriched, y_enriched, groups=groups_enriched))

scaler_enriched = StandardScaler()
X_enr_train = scaler_enriched.fit_transform(X_enriched[train_idx_enr])
X_enr_test = scaler_enriched.transform(X_enriched[test_idx_enr])
y_enr_train, y_enr_test = y_enriched[train_idx_enr], y_enriched[test_idx_enr]

print(f"Train: {len(X_enr_train)} frames ({len(set(groups_enriched[train_idx_enr]))} subjects)")
print(f"Test:  {len(X_enr_test)} frames ({len(set(groups_enriched[test_idx_enr]))} subjects)")


### Experiment 1 — RandomForest on All Enriched Features

Same regularized hyperparameters as the tuned baseline above (`max_depth`/`min_samples_leaf` capped, not the original unconstrained `max_depth=None`), just given the full enriched feature set instead of the curated 7.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Reuse the tuned hyperparameters from the earlier RandomizedSearchCV if it ran in this session;
# otherwise fall back to reasonable regularized defaults so this cell doesn't hard-depend on it.
rf_params = search.best_params_ if 'search' in globals() else {
    'n_estimators': 400, 'max_depth': 10, 'min_samples_leaf': 20, 'min_samples_split': 10, 'max_features': 'sqrt',
}

rf_all_features = RandomForestClassifier(**rf_params, class_weight='balanced', random_state=42, n_jobs=-1)
rf_all_features.fit(X_enr_train, y_enr_train)

y_pred_all = rf_all_features.predict(X_enr_test)
print("--- RandomForest, all enriched features ---")
print(classification_report(y_enr_test, y_pred_all, target_names=CLASS_NAMES))
print(f"Overall Accuracy: {accuracy_score(y_enr_test, y_pred_all):.4f}")

cm_all = confusion_matrix(y_enr_test, y_pred_all)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_all, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.title('Confusion Matrix (RandomForest, All Enriched Features)')
plt.show()


### Experiment 2 — Swap to HistGradientBoostingClassifier

Built into scikit-learn (no new dependency). Same enriched feature set and split, so this is a direct apples-to-apples comparison against Experiment 1.


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_model = HistGradientBoostingClassifier(
    max_depth=10,
    learning_rate=0.05,
    max_iter=300,
    l2_regularization=1.0,
    class_weight='balanced',
    random_state=42,
)
hgb_model.fit(X_enr_train, y_enr_train)

y_pred_hgb = hgb_model.predict(X_enr_test)
print("--- HistGradientBoostingClassifier, all enriched features ---")
print(classification_report(y_enr_test, y_pred_hgb, target_names=CLASS_NAMES))
print(f"Overall Accuracy: {accuracy_score(y_enr_test, y_pred_hgb):.4f}")

cm_hgb = confusion_matrix(y_enr_test, y_pred_hgb)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_hgb, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.title('Confusion Matrix (HistGradientBoosting, All Enriched Features)')
plt.show()


### Experiment 3 — SMOTE-Augmented Training

`SMOTE` (Synthetic Minority Oversampling) generates synthetic rows by interpolating between
nearby real samples of the same class. Applied to the **training set only** — SMOTE-ing the test
set would evaluate against fabricated data instead of real held-out subjects, silently inflating
the reported score. Classes here are already fairly balanced (this isn't primarily an imbalance
fix), so treat this as a robustness/decision-boundary-smoothing experiment on top of whichever of
Experiment 1/2 scored better, not a guaranteed improvement.


In [ ]:
!pip install -q imbalanced-learn


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_smote_train, y_smote_train = smote.fit_resample(X_enr_train, y_enr_train)

print(f"Before SMOTE: {len(X_enr_train)} rows, class counts: {dict(zip(*np.unique(y_enr_train, return_counts=True)))}")
print(f"After SMOTE:  {len(X_smote_train)} rows, class counts: {dict(zip(*np.unique(y_smote_train, return_counts=True)))}")

# Retrain the better of the two Experiment 1/2 models (by macro F1) on the SMOTE-resampled data.
from sklearn.metrics import f1_score
better_is_hgb = f1_score(y_enr_test, y_pred_hgb, average='macro') > f1_score(y_enr_test, y_pred_all, average='macro')

if better_is_hgb:
    print("\nRetraining HistGradientBoosting on SMOTE-resampled data (it scored higher above)...")
    smote_model = HistGradientBoostingClassifier(
        max_depth=10, learning_rate=0.05, max_iter=300, l2_regularization=1.0, random_state=42,
    )
else:
    print("\nRetraining RandomForest on SMOTE-resampled data (it scored higher above)...")
    smote_model = RandomForestClassifier(**rf_params, random_state=42, n_jobs=-1)

smote_model.fit(X_smote_train, y_smote_train)
y_pred_smote = smote_model.predict(X_enr_test)

print("\n--- SMOTE-augmented training ---")
print(classification_report(y_enr_test, y_pred_smote, target_names=CLASS_NAMES))
print(f"Overall Accuracy: {accuracy_score(y_enr_test, y_pred_smote):.4f}")

cm_smote = confusion_matrix(y_enr_test, y_pred_smote)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_smote, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.title('Confusion Matrix (SMOTE-Augmented)')
plt.show()


### Reading All of This Together

Line up, in order: original curated-7-feature baseline → `rf_tuned`/`rf_subject_norm`
(regularization/normalization diagnostics) → `rf_all_features` and `hgb_model` (enriched
features + algorithm swap) → `smote_model` (augmented training). The size of each jump tells you
which lever actually mattered:

* A **big jump from curated-7 to all-enriched-features** confirms the rolling/temporal columns
  are carrying real signal the raw single-frame features didn't have — the single most important
  thing to report, since it validates the whole enrichment approach.
* A **modest jump from RF to HistGradientBoosting** on the *same* features is the ceiling of
  "better algorithm, same information" — expect this to be smaller than the enrichment jump.
* A **small or no jump from SMOTE** is expected and fine — it was never meant to add information,
  just smooth the decision boundary.

Whatever the final number lands at, it's now backed by a real progression of controlled
experiments rather than a single tuning pass — that progression itself (what helped, what didn't,
and why) is worth reporting in the titulación document alongside the raw accuracy figures.
